In [48]:
import re
import spacy
import pandas as pd
from gensim import corpora
from gensim.models import CoherenceModel
from gensim.models.ldamodel import LdaModel
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from IPython.display import display

In [23]:
# Download VADER lexicon if not already downloaded
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/madhusudhananjeyaram/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [24]:
def load_chat_data(filepath):
    """
    Loads chat data from a text file, skipping bad lines.
    """
    df = pd.read_csv(filepath, header=None, encoding='utf8', on_bad_lines='skip')
    return df

In [25]:
def preprocess_chat_dataframe(df):
    """
    Cleans and structures the chat DataFrame.
    Drops the first row, renames columns, splits chat into time, name, and message.
    Returns the processed DataFrame.
    """
    df = df.drop(0)
    df.columns = ['Date', 'Chat']
    Message = df["Chat"].str.split("-", n=1, expand=True)
    df["Time"] = Message[0]
    Message1 = Message[1].str.split(":", n=1, expand=True)
    df["Name"] = Message1[0]
    df["Chat"] = Message1[1]
    df = df[["Date", "Time", "Name", "Chat"]]
    return df


In [42]:
def tfidf_lda_topic_modeling(df, n_topics=5, n_words=10):
    """
    Assigns a topic_type to each row in the DataFrame using LDA.
    Removes top_words from the output.
    Returns the DataFrame with a topic_type column.
    """
    
    custom_stop_words = list(ENGLISH_STOP_WORDS) + ['hi', 'hello', 'please', 'thanks']
    texts = df['Chat'].astype(str).tolist()
    vectorizer = CountVectorizer(stop_words=custom_stop_words)
    X = vectorizer.fit_transform(texts)
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
    lda.fit(X)

    # Assign topic_type to each row based on the highest topic probability
    topic_assignments = lda.transform(X).argmax(axis=1)
    df = df.copy()
    df['topic_type'] = ['Topic {}'.format(i+1) for i in topic_assignments]
    return df
  

In [27]:
def append_sentiment_scores(df, text_column='Chat'):
    """
    Appends sentiment scores (neg, neu, pos, compound) to the DataFrame for the specified text column.
    """
    sid = SentimentIntensityAnalyzer()
    sentiments = df[text_column].astype(str).apply(sid.polarity_scores)
    df['neg'] = sentiments.apply(lambda x: x['neg'])
    df['neu'] = sentiments.apply(lambda x: x['neu'])
    df['pos'] = sentiments.apply(lambda x: x['pos'])
    df['compound'] = sentiments.apply(lambda x: x['compound'])
    return df

In [44]:
def count_topic_types(df):
    """
    Counts the number of rows for each topic_type in the DataFrame.
    Returns a Series with topic_type as index and counts as values.
    """
    return df['topic_type'].value_counts()

## 1. Load data from file.


In [28]:
df = load_chat_data("mabel.txt")
print (df)

           0                                                  1
0   05/12/19   1:42 pm - Messages to this chat and calls are...
1   05/12/19   1:42 pm - Mabel Infoziant: Hi this is Mabel w...
2   05/12/19   1:42 pm - Mabel Infoziant: What’s your full name
3   05/12/19                      1:42 pm - AR❤: Ramisha Rani K
4   05/12/19                      1:42 pm - Mabel Infoziant: Ok
5   05/12/19   1:42 pm - Mabel Infoziant: ramisharanik@gmail...
6   05/12/19          1:43 pm - Mabel Infoziant: Your email Id?
7   05/12/19                             1:43 pm - AR❤: Yes Mam
8   05/12/19   1:43 pm - Mabel Infoziant: I will send 2 abst...
9   05/12/19                            1:43 pm - AR❤: Yeah mam
10  05/12/19   1:43 pm - Mabel Infoziant: Give me the list t...
11  05/12/19   1:43 pm - Mabel Infoziant: Send me cbe office...
12  05/12/19   1:44 pm - AR❤: Yeah Mam within Evening I will...
13  05/12/19   1:45 pm - AR❤: Tomorrow I have to visit the o...
14  05/12/19                      1:48 p

## 2. Preprocess (clean and structure) the chat data.


In [29]:
df = preprocess_chat_dataframe(df)

In [30]:
df

,Date,Time,Name,Chat
1,05/12/19,1:42 pm,Mabel Infoziant,Hi this is Mabel we just spoke
2,05/12/19,1:42 pm,Mabel Infoziant,What’s your full name
3,05/12/19,1:42 pm,AR❤,Ramisha Rani K
4,05/12/19,1:42 pm,Mabel Infoziant,Ok
5,05/12/19,1:42 pm,Mabel Infoziant,ramisharanik@gmail.com
6,05/12/19,1:43 pm,Mabel Infoziant,Your email Id?
7,05/12/19,1:43 pm,AR❤,Yes Mam
8,05/12/19,1:43 pm,Mabel Infoziant,I will send 2 abstracts for u to start working
9,05/12/19,1:43 pm,AR❤,Yeah mam
10,05/12/19,1:43 pm,Mabel Infoziant,Give me the list that u have too


In [31]:
df = append_sentiment_scores(df)

In [32]:
df

,Date,Time,Name,Chat,neg,neu,pos,compound
1,05/12/19,1:42 pm,Mabel Infoziant,Hi this is Mabel we just spoke,0.000,1.000,0.000,0.0000
2,05/12/19,1:42 pm,Mabel Infoziant,What’s your full name,0.000,1.000,0.000,0.0000
3,05/12/19,1:42 pm,AR❤,Ramisha Rani K,0.000,1.000,0.000,0.0000
4,05/12/19,1:42 pm,Mabel Infoziant,Ok,0.000,0.000,1.000,0.2960
5,05/12/19,1:42 pm,Mabel Infoziant,ramisharanik@gmail.com,0.000,1.000,0.000,0.0000
6,05/12/19,1:43 pm,Mabel Infoziant,Your email Id?,0.000,1.000,0.000,0.0000
7,05/12/19,1:43 pm,AR❤,Yes Mam,0.000,0.270,0.730,0.4019
8,05/12/19,1:43 pm,Mabel Infoziant,I will send 2 abstracts for u to start working,0.000,1.000,0.000,0.0000
9,05/12/19,1:43 pm,AR❤,Yeah mam,0.000,0.312,0.688,0.2960
10,05/12/19,1:43 pm,Mabel Infoziant,Give me the list that u have too,0.000,1.000,0.000,0.0000


## 3. Vectorize text into a document-term matrix.


In [43]:
topics_df = tfidf_lda_topic_modeling(df, n_topics=5, n_words=10)
topics_df

,Date,Time,Name,Chat,neg,neu,pos,compound,topic_type
1,05/12/19,1:42 pm,Mabel Infoziant,Hi this is Mabel we just spoke,0.000,1.000,0.000,0.0000,Topic 4
2,05/12/19,1:42 pm,Mabel Infoziant,What’s your full name,0.000,1.000,0.000,0.0000,Topic 1
3,05/12/19,1:42 pm,AR❤,Ramisha Rani K,0.000,1.000,0.000,0.0000,Topic 4
4,05/12/19,1:42 pm,Mabel Infoziant,Ok,0.000,0.000,1.000,0.2960,Topic 5
5,05/12/19,1:42 pm,Mabel Infoziant,ramisharanik@gmail.com,0.000,1.000,0.000,0.0000,Topic 4
6,05/12/19,1:43 pm,Mabel Infoziant,Your email Id?,0.000,1.000,0.000,0.0000,Topic 2
7,05/12/19,1:43 pm,AR❤,Yes Mam,0.000,0.270,0.730,0.4019,Topic 2
8,05/12/19,1:43 pm,Mabel Infoziant,I will send 2 abstracts for u to start working,0.000,1.000,0.000,0.0000,Topic 3
9,05/12/19,1:43 pm,AR❤,Yeah mam,0.000,0.312,0.688,0.2960,Topic 5
10,05/12/19,1:43 pm,Mabel Infoziant,Give me the list that u have too,0.000,1.000,0.000,0.0000,Topic 2


## 4.Display top words for each topic.

In [49]:
topic_counts = count_topic_types(topics_df)
display(topic_counts) 

topic_type
Topic 5    19
Topic 2    13
Topic 4     8
Topic 1     6
Topic 3     4
Name: count, dtype: int64